# Preprocessing: from two reference structures to DFT inputs

This tutorial prepares the DFT runs that berry-flux-diag needs, for BaTiO₃:

1. Set up the polar and non-polar reference structures.
2. Translate the non-polar structure so atoms move as little as possible, and
   interpolate images between the two if they are still too far apart.
3. Write Quantum ESPRESSO and VASP inputs for each structure.
4. Optionally, compute a k-point list on the irreducible Brillouin zone (IBZ)
   that both structures share, to make the DFT runs cheaper.

The postprocessing tutorial (`postprocess_DFT_files.ipynb`) then computes the
polarization from the finished runs.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by pymatgen

import numpy as np
from pymatgen.core import Lattice, Structure

import berry_flux_diag as bfd
from berry_flux_diag import symmetry

bfd.configure_logging();  # show progress messages; bfd.configure_logging("WARNING") to silence them

### Step 0: Relax the two reference structures

The polar and non-polar structures must already be relaxed, and must list the
same atoms in the same order: site *i* of one is paired with site *i* of the
other.

### Step 1: Set up the reference structures as pymatgen `Structure`s

In [2]:
material = "BaTiO3"
species = ["Ba", "Ti", "O", "O", "O"]  # order must match the coordinates

# non-polar (cubic) BaTiO3
np_orig_struct = Structure(
    Lattice([[4.035583, 0, 0], [0, 4.035583, 0], [0, 0, 4.035583]]),
    species,
    [[0.0, 0.0, 0.0028],   # Ba
     [0.5, 0.5, 0.5028],   # Ti
     [0.0, 0.5, 0.5028],   # O
     [0.5, 0.0, 0.5028],   # O
     [0.5, 0.5, 0.0028]],  # O
)

# polar (tetragonal) BaTiO3
pol_orig_struct = Structure(
    Lattice([[4.002006, 0, 0], [0, 4.002006, 0], [0, 0, 4.215507]]),
    species,
    [[0.0, 0.0, 0.020229],   # Ba
     [0.5, 0.5, 0.538921],   # Ti
     [0.0, 0.5, 0.492023],   # O
     [0.5, 0.0, 0.492023],   # O
     [0.5, 0.5, 0.970803]],  # O
)

### Step 2: Translate, and interpolate if needed

`preprocess_structs` finds the rigid translation of the non-polar structure
that minimizes the largest atomic displacement between the two. If that
displacement is still above `max_disp`, it inserts enough interpolated images
that no atom moves more than `max_disp` between adjacent structures. Each
adjacent pair then stays on one branch of the Berry phase.

In [3]:
structs_dict = bfd.preprocess.preprocess_structs(
    pol_orig_struct, np_orig_struct,
    translate=True,      # apply the translation that minimizes the largest displacement
    num_interps="auto",  # add as many images as max_disp requires
    max_disp=0.3,        # Angstrom, between corresponding atoms of adjacent structures
)
structs = structs_dict["structs"]

print(f"largest displacement as given:      {structs_dict['orig_max_disp']:.4f} A")
print(f"largest displacement translated:    {structs_dict['trans_max_disp']:.4f} A")
print(f"largest displacement between steps: {structs_dict['adj_max_disp']:.4f} A")
print(f"structures to run: {len(structs)}")

INFO berry_flux_diag.preprocess: translation [-0. -0. -0.] reduced the maximum displacement from 0.1523 to 0.1523 Angstrom


largest displacement as given:      0.1523 A
largest displacement translated:    0.1523 A
largest displacement between steps: 0.1523 A
structures to run: 2


For BaTiO₃ the translated structures are already within 0.3 Å, so only the
two end points need DFT runs.

When the structures are further apart than `max_disp`, interpolated images are
added. As an illustration, the same pair with a stricter `max_disp` of 0.1 Å
needs images in between, and each one needs its own DFT run:

In [4]:
strict = bfd.preprocess.preprocess_structs(pol_orig_struct, np_orig_struct, max_disp=0.1)
print(f"{len(strict['structs'])} structures, at most {strict['adj_max_disp']:.4f} A apart")

INFO berry_flux_diag.preprocess: translation [-0. -0. -0.] reduced the maximum displacement from 0.1523 to 0.1523 Angstrom


3 structures, at most 0.0761 A apart


The postprocessing tutorial shows how the polarization is summed along such
a chain. The rest of this tutorial keeps the two-structure BaTiO₃ pair.

### Step 3a (Quantum ESPRESSO): write `scf.in` files

One input per structure, named `pol_orig`, `interp_1`, …, `np_trans`.
Symmetry is switched off (`nosym`, `noinv`) so the wavefunctions cover the
full Brillouin zone. Set `pseudo_dir` to wherever your ONCV pseudopotentials
are.

In [5]:
qe_dir = "QE_io_files/"
pseudo_dir = "/global/homes/p/pabigail/qe_pseudos/"  # ONCV pseudopotentials
kpoints_grid = (5, 5, 5)
kshift = (0, 0, 0)
species_dict = {"species": ["Ba", "O", "Ti"],
                "masses": [137.3270, 15.9994, 47.8670],
                "pseudos": ["Ba_ONCV_PBE-1.0.upf",
                            "O_ONCV_PBE-1.0.upf",
                            "Ti_ONCV_PBE-1.0.upf"]}

bfd.preprocess.write_qe_in_scf_files(qe_dir, structs, material, pseudo_dir,
                                     kpoints_grid, kshift, species_dict)

### Step 3b (VASP): write `POSCAR` files

In [6]:
vasp_dir = "VASP_io_files/"
bfd.preprocess.write_vasp_in_poscar_files(vasp_dir, structs, material)

INFO berry_flux_diag.preprocess: Wrote VASP_io_files/POSCAR_BaTiO3_pol_orig


INFO berry_flux_diag.preprocess: Wrote VASP_io_files/POSCAR_BaTiO3_np_trans


For the VASP runs, the `INCAR` must contain

```
ISYM = -1      # full Brillouin zone, no symmetry reduction
LWAVE = .TRUE. # write the WAVECAR
```

and every structure must use the **same `KPOINTS`** mesh. Also pin `NBANDS`
to the same value for all structures. For a spin-polarized run, add
`ISPIN = 2`, the `MAGMOM`s, and `NUPDOWN` if the total moment is fixed.

### Step 4 (optional): run on the irreducible Brillouin zone instead

The DFT runs can be made cheaper by using only the irreducible Brillouin zone
(IBZ). berry-flux-diag unfolds the states to the full mesh when it reads
them. Both structures must use one identical k-point list, reduced with only
the symmetry operations they *share*, so each code's own automatic reduction
cannot be used. Instead, run in two steps:

1. **SCF** with symmetry on, on the full mesh, as usual.
2. **NSCF** on the shared IBZ list below, with symmetry off so the list is
   kept as given:
   - **VASP:** `ICHARG = 11`, `ISYM = 0`, `LWAVE = .TRUE.`, and the
     `KPOINTS` file below.
   - **Quantum ESPRESSO:** `calculation = 'nscf'`, `nosym = .true.`,
     `noinv = .true.`, and the `K_POINTS` card below.

For a spin-polarized run, pass each structure's collinear moments to
`shared_symmetry` (`pol_magmoms=`, `np_magmoms=`).

In [7]:
pol, non_pol = structs[0], structs[-1]
shared = symmetry.shared_symmetry(pol, non_pol)
print(f"space groups: {shared.space_groups}; {len(shared)} shared operations")

mesh = symmetry.monkhorst_pack(kpoints_grid, shift=kshift)  # the full mesh
reps, mult = symmetry.reduce_mesh(mesh, shared.k_rotations)
print(f"{len(mesh)} k-points in the full mesh -> {len(reps)} in the shared IBZ")

space groups: {'pol': 'P4mm (99)', 'np': 'Pm-3m (221)'}; 8 shared operations
125 k-points in the full mesh -> 18 in the shared IBZ


In [8]:
print(symmetry.vasp_kpoints_text(mesh[reps], mult))  # VASP KPOINTS file for the NSCF step

IBZ shared by the polar and non-polar structures
18
Reciprocal
0.000000000000 0.000000000000 0.000000000000 1
0.000000000000 0.000000000000 0.200000000000 2
0.000000000000 0.000000000000 0.400000000000 2
0.000000000000 0.200000000000 0.000000000000 4
0.000000000000 0.200000000000 0.200000000000 8
0.000000000000 0.200000000000 0.400000000000 8
0.000000000000 0.400000000000 0.000000000000 4
0.000000000000 0.400000000000 0.200000000000 8
0.000000000000 0.400000000000 0.400000000000 8
0.200000000000 0.200000000000 0.000000000000 4
0.200000000000 0.200000000000 0.200000000000 8
0.200000000000 0.200000000000 0.400000000000 8
0.200000000000 0.400000000000 0.000000000000 8
0.200000000000 0.400000000000 0.200000000000 16
0.200000000000 0.400000000000 0.400000000000 16
0.400000000000 0.400000000000 0.000000000000 4
0.400000000000 0.400000000000 0.200000000000 8
0.400000000000 0.400000000000 0.400000000000 8



In [9]:
print(symmetry.qe_kpoints_text(mesh[reps], mult))  # QE K_POINTS card for the NSCF step

K_POINTS crystal
  18
  0.0000000000 0.0000000000 0.0000000000 1
  0.0000000000 0.0000000000 0.2000000000 2
  0.0000000000 0.0000000000 0.4000000000 2
  0.0000000000 0.2000000000 0.0000000000 4
  0.0000000000 0.2000000000 0.2000000000 8
  0.0000000000 0.2000000000 0.4000000000 8
  0.0000000000 0.4000000000 0.0000000000 4
  0.0000000000 0.4000000000 0.2000000000 8
  0.0000000000 0.4000000000 0.4000000000 8
  0.2000000000 0.2000000000 0.0000000000 4
  0.2000000000 0.2000000000 0.2000000000 8
  0.2000000000 0.2000000000 0.4000000000 8
  0.2000000000 0.4000000000 0.0000000000 8
  0.2000000000 0.4000000000 0.2000000000 16
  0.2000000000 0.4000000000 0.4000000000 16
  0.4000000000 0.4000000000 0.0000000000 4
  0.4000000000 0.4000000000 0.2000000000 8
  0.4000000000 0.4000000000 0.4000000000 8

